# Decision trees for classification

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
A tree splits the data with yes/no questions so that each leaf becomes as **pure** as possible (all one class). A leaf predicts its majority class, and its probabilities are the class shares in the leaf.

> [A depth-2 tree on the Iris data. gini is the impurity, samples the count and value the count of each class in the node.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/tree_cls_struct.png)

*A depth-2 tree on the Iris data. gini is the impurity, samples the count and value the count of each class in the node.*

## Impurity measures
For a node where class $k$ has share $p_k$:

| Measure | Formula | Pure node | Maximum (two classes) |
|---|---|---|---|
| Gini | $1-\sum_k p_k^2$ | 0 | 0.5 |
| Entropy | $-\sum_k p_k\log_2p_k$ | 0 | 1 |

> [Impurity of a two-class node as a function of the share of class 1.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/impurity.png)

*Impurity of a two-class node as a function of the share of class 1.*

## Choosing the best split
The chosen split minimizes the **weighted average impurity** of the children (equivalently maximizes the **information gain**):

$$\text{Gain}=G(\text{parent})-\frac{n_L}{n}G(L)-\frac{n_R}{n}G(R)$$

## A worked example
10 samples: 5 healthy and 5 sick. Parent Gini $=1-0.5^2-0.5^2=0.5$.

| Split | Left | Right | Weighted Gini |
|---|---|---|---|
| age ≤ 40 | 4 healthy, 1 sick (Gini=0.32) | 1 healthy, 4 sick (0.32) | 0.32 |
| pressure ≤ 12 | 3 healthy, 2 sick (0.48) | 2 healthy, 3 sick (0.48) | 0.48 |

The "age" split is better: gain $0.5-0.32=0.18$.



In [ ]:
import numpy as np

def gini(counts):
    p = np.array(counts) / sum(counts)
    return 1 - np.sum(p ** 2)

def weighted(left, right):
    n = sum(left) + sum(right)
    return sum(left) / n * gini(left) + sum(right) / n * gini(right)

print("parent gini:", gini([5, 5]))
print("split on age     :", round(weighted([4, 1], [1, 4]), 3))
print("split on pressure:", round(weighted([3, 2], [2, 3]), 3))
entropy = lambda c: -sum(q * np.log2(q) for q in np.array(c) / sum(c) if q > 0)
print("entropy of [5, 5]:", entropy([5, 5]), " of [4, 1]:", round(entropy([4, 1]), 3))



## Depth and overfitting
> [Depth 1: one cut. Depth 4: good. No limit: every noisy point gets its own region.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/tree_cls_depth.png)

*Depth 1: one cut. Depth 4: good. No limit: every noisy point gets its own region.*

[[lab:cls_tree]]

In the lab, raise `max_depth` until the train accuracy reaches 1, then look at the test accuracy.

## Pruning
| Method | Parameters |
|---|---|
| Pre-pruning: stop earlier | `max_depth`, `min_samples_leaf`, `max_leaf_nodes` |
| Post-pruning: grow the full tree, then cut weak branches | `ccp_alpha` |



In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_tr, y_tr)
for a in path.ccp_alphas[::4]:
    t = DecisionTreeClassifier(ccp_alpha=a, random_state=0).fit(X_tr, y_tr)
    print(f"ccp_alpha={a:.4f}  leaves={t.get_n_leaves():2d}  train={t.score(X_tr, y_tr):.3f}  test={t.score(X_te, y_te):.3f}")



| Pros | Cons |
|---|---|
| Readable as "if-then" rules | Unstable |
| No scaling needed | Axis-parallel step boundaries |
| Feature importance | Overfits quickly |

### Exercise

Compute the entropy and the information gain of the 'age' split in the example.

In [ ]:
# Your answer here

### Exercise

Print the rules of a depth-3 tree on Iris with export_text.

In [ ]:
# Your answer here